In [14]:
import pandas as pd
import numpy as np
from pathlib import Path
from zipfile import ZipFile


# Open ZIP file
with ZipFile("RTH.zip") as z:

# Read April 1 market data
    with z.open("RTH/2025-04-01_assets.csv") as f:
        assets = pd.read_csv(f, index_col=0)

# Show first 5 rows
assets.head()

,SPX,ES_BID,ES_ASK,VIX
2025-04-01 09:31:21.491248,5590.32,5631.75,5632.00,22.66
2025-04-01 09:31:24.535800,5590.70,5631.75,5632.00,22.66
2025-04-01 09:31:27.582698,5589.06,5629.50,5629.75,22.66
2025-04-01 09:31:30.632772,5588.66,5630.50,5630.75,22.66
2025-04-01 09:31:33.638817,5588.82,5629.75,5630.00,22.69


In [15]:
# Calculate futures mid-price
assets["ES_MID"] = (assets["ES_BID"] + assets["ES_ASK"]) / 2

# Display first 5 rows
assets.head()

,SPX,ES_BID,ES_ASK,VIX,ES_MID
2025-04-01 09:31:21.491248,5590.32,5631.75,5632.00,22.66,5631.875
2025-04-01 09:31:24.535800,5590.70,5631.75,5632.00,22.66,5631.875
2025-04-01 09:31:27.582698,5589.06,5629.50,5629.75,22.66,5629.625
2025-04-01 09:31:30.632772,5588.66,5630.50,5630.75,22.66,5630.625
2025-04-01 09:31:33.638817,5588.82,5629.75,5630.00,22.69,5629.875


In [16]:
# Define observation date and expiry date
date = "2025-04-01"
expiry = "20250416" #Same format in the csv

In [17]:
with ZipFile("RTH.zip") as z:

    with z.open(f"RTH/{date}_put_bid_{expiry}.csv") as f:
        put_bid = pd.read_csv(f, index_col=0)

    with z.open(f"RTH/{date}_put_ask_{expiry}.csv") as f:
        put_ask = pd.read_csv(f, index_col=0)

    with z.open(f"RTH/{date}_call_bid_{expiry}.csv") as f:
        call_bid = pd.read_csv(f, index_col=0)

    with z.open(f"RTH/{date}_call_ask_{expiry}.csv") as f:
        call_ask = pd.read_csv(f, index_col=0)

# Display put bid prices
put_bid.head()

,4600.0,4625.0,4650.0,4675.0,4700.0,4725.0,4750.0,4775.0,4800.0,4825.0,...,6375.0,6400.0,6425.0,6450.0,6475.0,6500.0,6525.0,6550.0,6575.0,6600.0
2025-04-01 09:31:21.491248,3.2,3.3,3.5,3.6,3.8,4.0,4.2,4.4,4.6,4.8,...,764.2,789.1,814.6,839.0,863.9,888.9,913.8,938.7,963.7,988.6
2025-04-01 09:31:24.535800,3.2,3.3,3.5,3.6,3.8,4.0,4.2,4.4,4.6,4.8,...,764.2,789.1,814.6,839.0,863.9,888.9,913.8,938.7,963.7,988.6
2025-04-01 09:31:27.582698,3.2,3.3,3.5,3.6,3.8,4.0,4.2,4.4,4.6,4.8,...,767.0,791.9,816.9,841.8,866.7,891.7,916.6,941.5,966.5,991.4
2025-04-01 09:31:30.632772,3.2,3.3,3.5,3.6,3.8,4.0,4.2,4.4,4.6,4.8,...,766.6,791.5,816.4,841.4,866.3,891.2,916.2,941.1,966.0,991.0
2025-04-01 09:31:33.638817,3.2,3.3,3.5,3.6,3.8,4.0,4.2,4.4,4.6,4.8,...,765.1,791.3,816.2,841.2,866.1,891.0,916.0,939.7,964.6,990.8


In [18]:
# Convert index to datetime
put_bid.index = pd.to_datetime(put_bid.index)

# Select April 1 at 3:55 PM
target_time = pd.Timestamp("2025-04-01 15:55:00")

# Find the nearest available observation
i = put_bid.index.get_indexer([target_time], method="nearest")[0]

# Extract the selected time
selected_time = put_bid.index[i]

print("Selected time:", selected_time)

Selected time: 2025-04-01 15:55:01.126139


In [19]:
# Show put bids at selected time
put_bid.iloc[i].head()

4600.0    2.50
4625.0    2.65
4650.0    2.75
4675.0    2.85
4700.0    3.00
Name: 2025-04-01 15:55:01.126139, dtype: float64

In [20]:
# Select bid and ask prices at our chosen time
pb = put_bid.loc[str(selected_time)]
pa = put_ask.loc[str(selected_time)]
cb = call_bid.loc[str(selected_time)]
ca = call_ask.loc[str(selected_time)]

# Calculate option mid-prices
put_mid = (pb + pa) / 2
call_mid = (cb + ca) / 2

In [21]:
# Create a table
options = pd.DataFrame({
    "Put_Mid": put_mid,
    "Call_Mid": call_mid
})

options.index.name = "Strike"

# Display first 5 rows
options.head()

,Put_Mid,Call_Mid
Strike,,
4600.0,2.600,1043.15
4625.0,2.725,1018.20
4650.0,2.850,993.55
4675.0,2.975,968.50
4700.0,3.100,943.95


In [22]:
# Define option expiry
expiry_time = pd.Timestamp("2025-04-16 16:00:00")

# Calculate time to expiry in years
T = (expiry_time - selected_time).total_seconds() / (365 * 24 * 60 * 60)

# Assume annual risk-free rate of 4.3%
r = 0.043

# Calculate discount factor
D = np.exp(-r * T)

print("Time to expiry:", T)
print("Discount factor:", D)

Time to expiry: 0.041105367638920594
Discount factor: 0.9982340303541694


In [23]:
# Make sure strike prices are numbers
options.index = options.index.astype(float)

# Keep options with positive mid-prices
valid = options[(options["Put_Mid"] > 0) & (options["Call_Mid"] > 0)]

# Choose the strike where call and put prices are closest
K = (valid["Call_Mid"] - valid["Put_Mid"]).abs().idxmin()

# Get option prices at that strike
C = valid.loc[K, "Call_Mid"]
P = valid.loc[K, "Put_Mid"]

# Calculate forward price using put-call parity
F = K + (C - P) / D

print("Strike:", K)
print("Call Mid:", C)
print("Put Mid:", P)
print("Forward Price:", F)

Strike: 5650.0
Call Mid: 89.8
Put Mid: 95.35
Forward Price: 5644.440181529344


In [24]:
from scipy.stats import norm
from scipy.optimize import brentq

# Black formula for a put option
def black_put(sigma):

    d1 = (np.log(F / K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    price = D * (K * norm.cdf(-d2) - F * norm.cdf(-d1))

    return price

In [25]:
# Find volatility that matches the market put price
iv_atm = brentq(lambda sigma: black_put(sigma) - P, 0.0001, 3.0)

In [26]:
print("ATM Implied Volatility:", iv_atm)
print("ATM Implied Volatility (%):", iv_atm * 100)
print("Calculated Put Price:", black_put(iv_atm))
print("Actual Put Price:", P)

ATM Implied Volatility: 0.2029895696024864
ATM Implied Volatility (%): 20.298956960248642
Calculated Put Price: 95.35000000000062
Actual Put Price: 95.35


In [27]:
# Calculate the target 95% strike
target_95 = 0.95 * F

In [28]:
# Find the nearest available strike
K_95 = min(options.index, key=lambda x: abs(x - target_95))

In [29]:
# Get the put mid-price at that strike
P_95 = options.loc[K_95, "Put_Mid"]

In [30]:
print("Target Strike:", target_95)
print("Selected Strike:", K_95)
print("Put Mid-Price:", P_95)

Target Strike: 5362.218172452876
Selected Strike: 5350.0
Put Mid-Price: 22.35


In [31]:
# Black formula for 95% put
def black_put_95(sigma):

    d1 = (np.log(F / K_95) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    price = D * (K_95 * norm.cdf(-d2) - F * norm.cdf(-d1))

    return price

In [33]:
# Find implied volatility
iv_95 = brentq(lambda sigma: black_put_95(sigma) - P_95, 0.0001, 3.0)

In [34]:
print("95% Put Implied Volatility (%):", iv_95 * 100)
print("Calculated Put Price:", black_put_95(iv_95))
print("Actual Put Price:", P_95)

95% Put Implied Volatility (%): 25.620500862391815
Calculated Put Price: 22.350000000007324
Actual Put Price: 22.35


In [35]:
# Calculate volatility skew
skew = iv_95 - iv_atm
print("Volatility Skew (% points):", skew * 100)

Volatility Skew (% points): 5.3215439021431745


In [36]:
# Calculate 105% strike
target_105 = 1.05 * F

In [37]:
# Find nearest available strike
K_105 = min(options.index, key=lambda x: abs(x - target_105))

In [38]:
# Get call mid-price
C_105 = options.loc[K_105, "Call_Mid"]

In [39]:
print("Target Strike:", target_105)
print("Selected Strike:", K_105)
print("Call Mid-Price:", C_105)

Target Strike: 5926.662190605812
Selected Strike: 5925.0
Call Mid-Price: 4.25


In [40]:
# Black formula for 105% call
def black_call_105(sigma):

    d1 = (np.log(F / K_105) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    price = D * (F * norm.cdf(d1) - K_105 * norm.cdf(d2))

    return price

In [41]:
# Find implied volatility
iv_105 = brentq(lambda sigma: black_call_105(sigma) - C_105, 0.0001, 3.0)

In [42]:
print("105% Call Implied Volatility (%):", iv_105 * 100)
print("Calculated Call Price:", black_call_105(iv_105))
print("Actual Call Price:", C_105)

105% Call Implied Volatility (%): 15.092972791712025
Calculated Call Price: 4.249999999940881
Actual Call Price: 4.25


In [43]:
# Create summary table for April 1
summary = pd.DataFrame({
    "Date": ["2025-04-01"],
    "Forward": [F],
    "ATM_IV": [iv_atm * 100],
    "Put_95_IV": [iv_95 * 100],
    "Call_105_IV": [iv_105 * 100],
    "Skew": [(iv_95 - iv_atm) * 100]
})

summary.round(2)

,Date,Forward,ATM_IV,Put_95_IV,Call_105_IV,Skew
0,2025-04-01,5644.44,20.3,25.62,15.09,5.32


In [44]:
# April 3
date_03 = "2025-04-03"
expiry_03 = "20250416"

In [45]:
with ZipFile("RTH.zip") as z:

    with z.open(f"RTH/{date_03}_put_bid_{expiry_03}.csv") as f:
        put_bid_03 = pd.read_csv(f, index_col=0)

    with z.open(f"RTH/{date_03}_put_ask_{expiry_03}.csv") as f:
        put_ask_03 = pd.read_csv(f, index_col=0)

    with z.open(f"RTH/{date_03}_call_bid_{expiry_03}.csv") as f:
        call_bid_03 = pd.read_csv(f, index_col=0)

    with z.open(f"RTH/{date_03}_call_ask_{expiry_03}.csv") as f:
        call_ask_03 = pd.read_csv(f, index_col=0)

In [46]:
# Convert April 3 timestamps to datetime
put_bid_03.index = pd.to_datetime(put_bid_03.index)

# Select April 3 at 10 AM
target_time_03 = pd.Timestamp("2025-04-03 10:00:00")

In [53]:
# Find the nearest observation
i_03 = put_bid_03.index.get_indexer([target_time_03], method = "nearest")[0]

# Save selected time
selected_time_03 = put_bid_03.index[i_03]

In [54]:
print("Selected time:", selected_time_03)
print("Number of strikes:", len(put_bid_03.columns))

Selected time: 2025-04-03 09:59:57.365402
Number of strikes: 81


In [55]:
# Select April 3 option prices
pb_03 = put_bid_03.loc[str(selected_time_03)]
pa_03 = put_ask_03.loc[str(selected_time_03)]
cb_03 = call_bid_03.loc[str(selected_time_03)]
ca_03 = call_ask_03.loc[str(selected_time_03)]

In [56]:
# Calculate mid-prices
put_mid_03 = (pb_03 + pa_03) / 2
call_mid_03 = (cb_03 + ca_03) / 2

In [57]:
# Create April 3 options table
options_03 = pd.DataFrame({
    "Put_Mid": put_mid_03.where((pb_03 >= 0) & (pa_03 >= 0)),
    "Call_Mid": call_mid_03.where((cb_03 >= 0) & (ca_03 >= 0))
})

options_03.index.name = "Strike"

options_03.head()

,Put_Mid,Call_Mid
Strike,,
4600.0,6.65,883.50
4625.0,5.90,858.80
4650.0,5.95,834.05
4675.0,4.90,809.35
4700.0,6.30,784.65


In [58]:
# April 3 expiry date
expiry_time_03 = pd.Timestamp("2025-04-16 16:00:00")

# Calculate time to expiry
T_03 = (expiry_time_03 - selected_time_03).total_seconds() / (365 * 24 * 60 * 60)

# Calculate discount factor
D_03 = np.exp(-r * T_03)

print("Time to Expiry:", T_03)
print("Discount Factor:", D_03)

Time to Expiry: 0.036301453405568236
Discount Factor: 0.9984402551718576


In [59]:
# Make sure strikes are numbers
options_03.index = options_03.index.astype(float)

In [61]:
# Keep valid option prices
valid_03 = options_03.dropna()
valid_03 = valid_03[(valid_03["Put_Mid"] > 0) & (valid_03["Call_Mid"] > 0)]

In [62]:
# Find strike where call and put prices are closest
K_03 = (valid_03["Call_Mid"] - valid_03["Put_Mid"]).abs().idxmin()

In [63]:
# Get call and put prices
C_03 = valid_03.loc[K_03, "Call_Mid"]
P_03 = valid_03.loc[K_03, "Put_Mid"]

In [64]:
# Calculate forward price
F_03 = K_03 + (C_03 - P_03) / D_03

In [72]:
print("Strike:", K_03)
print("Call Mid:", C_03)
print("Put Mid:", P_03)
print("Forward Price:", F_03)

Strike: 5475.0
Call Mid: 108.19999999999999
Put Mid: 103.25
Forward Price: 5479.9577327980915


In [73]:
# Black formula for both put and call options
def black_price(sigma, F, K, T, D, kind):

    d1 = (np.log(F / K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if kind == "put":
        price = D * (K * norm.cdf(-d2) - F * norm.cdf(-d1))

    elif kind == "call":
        price = D * (F * norm.cdf(d1) - K * norm.cdf(d2))

    return price

In [74]:
# Calculate implied volatility
def implied_vol(price, F, K, T, D, kind):

    iv = brentq(
        lambda sigma: black_price(sigma, F, K, T, D, kind) - price,
        0.0001, 3.0)

    return iv

In [75]:
# April 3 ATM implied volatility
iv_atm_03 = implied_vol(P_03, F_03, K_03, T_03, D_03, "put")

print("April 3 ATM IV (%):", iv_atm_03 * 100)

April 3 ATM IV (%): 25.431358811911398


In [76]:
# Calculate IV for any strike level
def strike_iv(options, F, T, D, ratio, kind):

    if kind == "put":
        prices = options["Put_Mid"]
    else:
        prices = options["Call_Mid"]

    # Keep valid prices
    prices = prices.dropna()
    prices = prices[prices > 0]

    # Find closest strike
    K = min(prices.index, key=lambda x: abs(x - ratio * F))

    # Calculate implied volatility
    iv = implied_vol(prices.loc[K], F, K, T, D, kind)

    return K, iv

In [77]:
# April 3 95% put
K_95_03, iv_95_03 = strike_iv(
    options_03, F_03, T_03, D_03, 0.95, "put"
)

# April 3 105% call
K_105_03, iv_105_03 = strike_iv(
    options_03, F_03, T_03, D_03, 1.05, "call"
)

# Calculate skew
skew_03 = iv_95_03 - iv_atm_03

In [78]:
print("95% Put IV (%):", iv_95_03 * 100)
print("105% Call IV (%):", iv_105_03 * 100)
print("Skew (percentage points):", skew_03 * 100)

95% Put IV (%): 31.82698131993555
105% Call IV (%): 20.794555819068954
Skew (percentage points): 6.395622508024151


In [79]:
# Create April 3 summary
april_03 = pd.DataFrame({
    "Date": ["2025-04-03"],
    "Forward": [F_03],
    "ATM_IV": [iv_atm_03 * 100],
    "Put_95_IV": [iv_95_03 * 100],
    "Call_105_IV": [iv_105_03 * 100],
    "Skew": [skew_03 * 100]
})

In [93]:
# Combine April 1 and April 3
summary = pd.concat([
    summary[summary["Date"] != "2025-04-03"],
    april_03
], ignore_index=True)

summary.round(2)

,Date,Forward,ATM_IV,Put_95_IV,Call_105_IV,Skew
0,2025-04-01,5644.44,20.30,25.62,15.09,5.32
1,2025-04-04,5089.18,45.47,53.59,38.54,8.12
2,2025-04-08,4994.83,41.08,46.21,36.17,5.13
3,2025-04-09,5478.36,25.39,30.63,20.47,5.24
4,2025-04-03,5479.96,25.43,31.83,20.79,6.40


In [82]:
#Create a reusable code to extract data instead of repeating all the code
def analyze_day(date, expiry, time):

    # 1. Load option data
    data = {}

    with ZipFile("RTH.zip") as z:

        for name in ["put_bid", "put_ask", "call_bid", "call_ask"]:

            file = f"RTH/{date}_{name}_{expiry}.csv"

            with z.open(file) as f:
                data[name] = pd.read_csv(f, index_col=0)

    # 2. Convert timestamps and strikes
    for df in data.values():
        df.index = pd.to_datetime(df.index)
        df.columns = df.columns.astype(float)

    # 3. Select observation time
    target = pd.Timestamp(f"{date} {time}")

    i = data["put_bid"].index.get_indexer(
        [target], method="nearest"
    )[0]

    selected_time = data["put_bid"].index[i]

    # 4. Calculate option mid-prices
    pb = data["put_bid"].loc[selected_time]
    pa = data["put_ask"].loc[selected_time]
    cb = data["call_bid"].loc[selected_time]
    ca = data["call_ask"].loc[selected_time]

    options = pd.DataFrame({
        "Put_Mid": ((pb + pa) / 2).where((pb >= 0) & (pa >= 0)),
        "Call_Mid": ((cb + ca) / 2).where((cb >= 0) & (ca >= 0))
    })

    options.index.name = "Strike"

    # 5. Calculate time to expiry
    expiry_time = pd.to_datetime(expiry, format="%Y%m%d")
    expiry_time = expiry_time + pd.Timedelta(hours=16)

    T = (expiry_time - selected_time).total_seconds() / (365 * 24 * 60 * 60)

    # 6. Calculate discount factor
    r = 0.043
    D = np.exp(-r * T)

    # 7. Calculate forward price
    valid = options.dropna()
    valid = valid[(valid["Put_Mid"] > 0) &
                  (valid["Call_Mid"] > 0)]

    K = (valid["Call_Mid"] - valid["Put_Mid"]).abs().idxmin()

    C = valid.loc[K, "Call_Mid"]
    P = valid.loc[K, "Put_Mid"]

    F = K + (C - P) / D

    # 8. Calculate implied volatilities
    iv_atm = implied_vol(P, F, K, T, D, "put")

    K_95, iv_95 = strike_iv(options, F, T, D, 0.95, "put")

    K_105, iv_105 = strike_iv(options, F, T, D, 1.05, "call")

    # 9. Calculate skew
    skew = iv_95 - iv_atm

    # 10. Create summary table
    result = pd.DataFrame({
        "Date": [date],
        "Forward": [F],
        "ATM_IV": [iv_atm * 100],
        "Put_95_IV": [iv_95 * 100],
        "Call_105_IV": [iv_105 * 100],
        "Skew": [skew * 100]
    })

    print("Selected time:", selected_time)

    return result, options

In [85]:
# Test our function on April 3
test_03, test_options_03 = analyze_day("2025-04-03", "20250416", "10:00")

test_03.round(2)

Selected time: 2025-04-03 09:59:57.365402


,Date,Forward,ATM_IV,Put_95_IV,Call_105_IV,Skew
0,2025-04-03,5479.96,25.43,31.83,20.79,6.4


In [84]:
# Analyze April 4
result_04, options_04 = analyze_day("2025-04-04", "20250416", "15:55")

result_04.round(2)

Selected time: 2025-04-04 15:55:00.149264


,Date,Forward,ATM_IV,Put_95_IV,Call_105_IV,Skew
0,2025-04-04,5089.18,45.47,53.59,38.54,8.12


In [86]:
# Analyze April 8
result_08, options_08 = analyze_day("2025-04-08", "20250515", "15:55")

result_08.round(2)

Selected time: 2025-04-08 15:54:58.674316


,Date,Forward,ATM_IV,Put_95_IV,Call_105_IV,Skew
0,2025-04-08,4994.83,41.08,46.21,36.17,5.13


In [88]:
# Analyze April 9
result_09, options_09 = analyze_day("2025-04-09", "20250515", "15:55")

result_09.round(2)

Selected time: 2025-04-09 15:54:59.061800


,Date,Forward,ATM_IV,Put_95_IV,Call_105_IV,Skew
0,2025-04-09,5478.36,25.39,30.63,20.47,5.24


In [95]:
# Rename April 3 results
result_03 = test_03

# Combine all five dates
summary = pd.concat([
    summary[summary["Date"] == "2025-04-01"],
    result_03,
    result_04,
    result_08,
    result_09], ignore_index = True)

summary.round(2)

,Date,Forward,ATM_IV,Put_95_IV,Call_105_IV,Skew
0,2025-04-01,5644.44,20.30,25.62,15.09,5.32
1,2025-04-03,5479.96,25.43,31.83,20.79,6.40
2,2025-04-04,5089.18,45.47,53.59,38.54,8.12
3,2025-04-08,4994.83,41.08,46.21,36.17,5.13
4,2025-04-09,5478.36,25.39,30.63,20.47,5.24


In [96]:
from lets_plot import *
LetsPlot.setup_html()

# Prepare data
chart_data = summary.copy()

# Convert data from wide to long format
chart_data = chart_data.melt(
    id_vars = "Date",
    value_vars = ["ATM_IV", "Put_95_IV", "Call_105_IV"],
    var_name = "Option",
    value_name = "IV"
)

# Create line chart
p = (ggplot(chart_data, aes(x = "Date", y = "IV", color = "Option", group = "Option"))
     + geom_line(size = 1)
     + geom_point(size = 3)
     + labs(
         title = "Implied Volatility During Liberation Day Tariffs",
         x = "Date",
         y = "Implied Volatility (%)",
         color = "Option Type"
     )
     + theme_minimal()
     + ggsize(850, 450)
)

p

In [97]:
# Get VIX for a selected date and time (VIX is in the csv file)
def get_vix(date, time):

    with ZipFile("RTH.zip") as z:
        file = f"RTH/{date}_assets.csv"

        with z.open(file) as f:
            assets_day = pd.read_csv(f, index_col=0)

    # Convert timestamps
    assets_day.index = pd.to_datetime(assets_day.index)

    # Find nearest observation
    target = pd.Timestamp(f"{date} {time}")
    i = assets_day.index.get_indexer([target], method="nearest")[0]

    # Retrieve VIX
    vix = assets_day.iloc[i]["VIX"]

    return vix

In [98]:
vix_01 = get_vix("2025-04-01", "15:55")
vix_03 = get_vix("2025-04-03", "10:00")

print("April 1 VIX:", vix_01)
print("April 3 VIX:", vix_03)

April 1 VIX: 21.85
April 3 VIX: 26.72


In [99]:
# Get VIX for April 4, 8 and 9
vix_04 = get_vix("2025-04-04", "15:55")
vix_08 = get_vix("2025-04-08", "15:55")
vix_09 = get_vix("2025-04-09", "15:55")

print("April 4 VIX:", vix_04)
print("April 8 VIX:", vix_08)
print("April 9 VIX:", vix_09)

April 4 VIX: 43.8
April 8 VIX: 55.22
April 9 VIX: 33.23


In [100]:
# Store VIX values
vix_values = {
    "2025-04-01": vix_01,
    "2025-04-03": vix_03,
    "2025-04-04": vix_04,
    "2025-04-08": vix_08,
    "2025-04-09": vix_09
}

# Add VIX to summary
summary["VIX"] = summary["Date"].map(vix_values)

# Display updated summary
summary.round(2)

,Date,Forward,ATM_IV,Put_95_IV,Call_105_IV,Skew,VIX
0,2025-04-01,5644.44,20.30,25.62,15.09,5.32,21.85
1,2025-04-03,5479.96,25.43,31.83,20.79,6.40,26.72
2,2025-04-04,5089.18,45.47,53.59,38.54,8.12,43.80
3,2025-04-08,4994.83,41.08,46.21,36.17,5.13,55.22
4,2025-04-09,5478.36,25.39,30.63,20.47,5.24,33.23


In [101]:
# Prepare data for plotting
chart_data = summary.melt(
    id_vars="Date",
    value_vars=["ATM_IV", "Put_95_IV", "Call_105_IV", "VIX"],
    var_name="Option",
    value_name="IV"
)

# Plot implied volatility and VIX
p_iv_vix = (ggplot(chart_data, aes(x="Date", y="IV", color="Option", group="Option"))
     + geom_line(size=1)
     + geom_point(size=3)
     + labs(
         title="Implied Volatility and VIX During Liberation Day Tariffs",
         x="Date",
         y="Volatility (%)",
         color="Volatility Measure"
     )
     + theme_minimal()
     + ggsize(850, 450)
)

p_iv_vix

In [102]:
# Calculate the volatility smile
def calculate_smile(options, F, T, D, date):

    results = []

    for K in options.index:

        # Select strikes between 85% and 110% of forward
        ratio = K / F

        if ratio < 0.85 or ratio > 1.10:
            continue

        # Use puts below forward and calls above forward
        if K < F:
            kind = "put"
            price = options.loc[K, "Put_Mid"]
        else:
            kind = "call"
            price = options.loc[K, "Call_Mid"]

        # Skip missing or invalid prices
        if pd.isna(price) or price <= 0:
            continue

        # Calculate implied volatility
        try:
            iv = implied_vol(price, F, K, T, D, kind)

            results.append({
                "Date": date,
                "Moneyness": ratio * 100,
                "IV": iv * 100
            })

        except ValueError:
            continue

    return pd.DataFrame(results)

In [103]:
# Calculate April 1 volatility smile
smile_01 = calculate_smile(
    options, F, T, D, "April 1"
)

# Display first 5 rows
smile_01.head()

,Date,Moneyness,IV
0,April 1,85.039434,40.100969
1,April 1,85.482348,39.334380
2,April 1,85.925262,38.466890
3,April 1,86.368175,37.660332
4,April 1,86.811089,36.833428


In [104]:
# April 4 forward price
F_04 = result_04["Forward"].iloc[0]

# April 4 selected time and expiry
selected_time_04 = pd.Timestamp("2025-04-04 15:55:00.149264")
expiry_time_04 = pd.Timestamp("2025-04-16 16:00:00")

# Calculate T and D
T_04 = (expiry_time_04 - selected_time_04).total_seconds() / (365 * 24 * 60 * 60)
D_04 = np.exp(-0.043 * T_04)

# Calculate April 4 volatility smile
smile_04 = calculate_smile(
    options_04, F_04, T_04, D_04, "April 4"
)

smile_04.head()

,Date,Moneyness,IV
0,April 4,90.387759,60.210620
1,April 4,90.878996,60.016591
2,April 4,91.370234,59.171418
3,April 4,91.861472,58.267281
4,April 4,92.352710,57.608227


In [105]:
# Combine the two smile tables
smile_all = pd.concat([smile_01, smile_04], ignore_index=True)

# Plot both smiles
p_smile = (ggplot(smile_all, aes(x="Moneyness", y="IV", color="Date", group="Date"))
    + geom_line(size=1)
    + geom_point(size=2)
    + labs(
        title="Volatility Smile: April 1 vs April 4",
        x="Moneyness (%)",
        y="Implied Volatility (%)",
        color="Date"
    )
    + theme_minimal()
    + ggsize(850, 450)
)

p_smile

In [106]:
# Calculate our own volatility index for April 1
K0_01 = options.index[options.index <= F].max()

# Select out-of-the-money option prices
otm_01 = options.copy()

otm_01["Price"] = np.where(
    otm_01.index < K0_01,
    otm_01["Put_Mid"],
    np.where(
        otm_01.index > K0_01,
        otm_01["Call_Mid"],
        (otm_01["Put_Mid"] + otm_01["Call_Mid"]) / 2
    )
)

# Remove invalid prices
otm_01 = otm_01[otm_01["Price"] > 0].copy()

# Calculate strike intervals
strikes = otm_01.index.to_numpy()

delta_k = np.empty(len(strikes))
delta_k[0] = strikes[1] - strikes[0]
delta_k[-1] = strikes[-1] - strikes[-2]
delta_k[1:-1] = (strikes[2:] - strikes[:-2]) / 2

otm_01["DeltaK"] = delta_k

# Calculate volatility using the VIX-style formula
total = (
    otm_01["DeltaK"] / strikes**2 * otm_01["Price"]
).sum()

variance_01 = (2 / (T * D)) * total - (F / K0_01 - 1)**2 / T

own_vix_01 = np.sqrt(variance_01) * 100

print("Our Volatility Index:", own_vix_01)
print("Official VIX:", vix_01)
print("Difference:", own_vix_01 - vix_01)

Our Volatility Index: 22.222619475518375
Official VIX: 21.85
Difference: 0.3726194755183734


In [107]:
# Calculate our own volatility index
def own_vol_index(options, F, T, D):

    # Find strike below the forward price
    K0 = options.index[options.index <= F].max()

    # Select out-of-the-money prices
    otm = options.copy()

    otm["Price"] = np.where(
        otm.index < K0,
        otm["Put_Mid"],
        np.where(
            otm.index > K0,
            otm["Call_Mid"],
            (otm["Put_Mid"] + otm["Call_Mid"]) / 2
        )
    )

    # Remove invalid prices
    otm = otm[otm["Price"] > 0].copy()
    otm = otm.sort_index()

    # Calculate strike intervals
    strikes = otm.index.to_numpy()

    delta_k = np.empty(len(strikes))
    delta_k[0] = strikes[1] - strikes[0]
    delta_k[-1] = strikes[-1] - strikes[-2]
    delta_k[1:-1] = (strikes[2:] - strikes[:-2]) / 2

    # Calculate variance
    total = (
        delta_k / strikes**2 * otm["Price"]
    ).sum()

    variance = (2 / (T * D)) * total - (F / K0 - 1)**2 / T

    # Convert variance to volatility percentage
    own_vix = np.sqrt(variance) * 100

    return own_vix

In [108]:
# Test our function
test_vix_01 = own_vol_index(options, F, T, D)

print("Calculated VIX:", test_vix_01)
print("Previous Result:", own_vix_01)

Calculated VIX: 22.222619475518375
Previous Result: 22.222619475518375


In [109]:
# Store the five observation details
moments = {
    "2025-04-01": (options, "2025-04-01 15:55:01.126139", "2025-04-16"),
    "2025-04-03": (options_03, "2025-04-03 09:59:57.365402", "2025-04-16"),
    "2025-04-04": (options_04, "2025-04-04 15:55:00.149264", "2025-04-16"),
    "2025-04-08": (options_08, "2025-04-08 15:54:58.674316", "2025-05-15"),
    "2025-04-09": (options_09, "2025-04-09 15:54:59.061800", "2025-05-15")
}

# Calculate our index for every date
own_vix_values = {}

for date, (opt, time, expiry) in moments.items():

    T_day = (
        pd.Timestamp(expiry + " 16:00") - pd.Timestamp(time)
    ).total_seconds() / (365 * 24 * 60 * 60)

    D_day = np.exp(-0.043 * T_day)

    F_day = summary.loc[summary["Date"] == date, "Forward"].iloc[0]

    own_vix_values[date] = own_vol_index(opt, F_day, T_day, D_day)

# Add results to our summary
summary["Own_VIX"] = summary["Date"].map(own_vix_values)

# Calculate difference from official VIX
summary["VIX_Difference"] = summary["Own_VIX"] - summary["VIX"]

summary.round(2)

,Date,Forward,ATM_IV,Put_95_IV,Call_105_IV,Skew,VIX,Own_VIX,VIX_Difference
0,2025-04-01,5644.44,20.30,25.62,15.09,5.32,21.85,22.22,0.37
1,2025-04-03,5479.96,25.43,31.83,20.79,6.40,26.72,27.33,0.61
2,2025-04-04,5089.18,45.47,53.59,38.54,8.12,43.80,44.65,0.85
3,2025-04-08,4994.83,41.08,46.21,36.17,5.13,55.22,35.31,-19.91
4,2025-04-09,5478.36,25.39,30.63,20.47,5.24,33.23,27.61,-5.62


In [110]:
# Create volatility index comparison table
vix_table = summary[
    ["Date", "Own_VIX", "VIX", "VIX_Difference"]
].copy()

vix_table.round(2)

,Date,Own_VIX,VIX,VIX_Difference
0,2025-04-01,22.22,21.85,0.37
1,2025-04-03,27.33,26.72,0.61
2,2025-04-04,44.65,43.80,0.85
3,2025-04-08,35.31,55.22,-19.91
4,2025-04-09,27.61,33.23,-5.62
